In [1]:
# ============================================================
# CRISP-DM Pipeline — Clustering
# Dataset  : Microsoft GUIDE Security Incident Prediction
#            (Kaggle: microsoft-security-incident-prediction)
# Train    : GUIDE_Train.csv (~2 GB, ~9.5M rows, 45 cols)
# Test     : GUIDE_Test.csv  (~1 GB, ~4.1M rows, 46 cols)
# Pipeline : clustering_pipeline_config_option_a.yml (drift-aware)
#            clustering_pipeline_config_option_b.yml (train-only)
# Stages   : 2-Understanding | 3-Preparation | 4-Modeling | 5-Evaluation
# ============================================================


In [1]:
from crispdm.api.execution_facade_api.py import (
    init_run_api,
    run_api_phase2_1,
    run_api_phase2_2,
    run_api_phase2_3,
    run_api_phase2_4,
)

from crispdm.core.paths_utils_core import find_project_root

In [2]:
# Project root paths resolved dynamically from pyproject.toml / .git
PROJECT_ROOT = find_project_root()
CONFIG_ROOT = PROJECT_ROOT / "config"
OUT_ROOT = PROJECT_ROOT / "out"

In [3]:
# Pipeline configuration
# Option A: combine_before_sampling=True  -> load_csv_combined -> drift detection active
# Option B: combine_before_sampling=False -> load_train_only   -> no drift detection
# Switch between options by changing the YAML filename only — no code changes needed.
pipeline_config_path = str(CONFIG_ROOT / "pipelines" /
                           "clustering_pipeline_config_option_a.yml")
dataset_config_path = str(CONFIG_ROOT / "datasets" / "dataset_config.yml")
dataset_key = "microsoft_security_incident"

# Runtime overrides injected into the YAML ${...} placeholders
# id_cols   : columns excluded from clustering features (record identifiers)
# output_root: root directory for all run artifacts
notebook_vars = {
    "id_cols": ["IncidentId", "AlertId", "DetectorId", "DeviceId"],
    "output_root": str(OUT_ROOT),
}

In [5]:
  # — Cell: INIT (builds config + run_dir) ——————
# Internally: detects task from YAML -> downloads raw data if needed
# -> builds ProjectConfig -> creates out/runs/{task}/{dataset}/{run_id}/
# -> returns ClusteringRunContext ready for Phase 2
ctx = init_run_api(
    pipeline_config_path=pipeline_config_path,
    dataset_config_path=dataset_config_path,
    dataset_key=dataset_key,
    notebook_vars=notebook_vars,
)
# ctx.task == ProblemType.CLUSTERING  <- determined by the .yml

DEBUG: Policy keys found: []


KeyError: 'save_all_as_png'

In [6]:
############ 2. PHASE 2: Data Understanding ############
# Stage contract: REPORT ONLY — data is never modified in Stage 2.
# All outputs are written to: out/runs/clustering/{dataset}/{run_id}/phase2_data_understanding/

In [7]:
# 2.1. Sub-phase: Initial Data Collection
# Option A: load_csv_combined(train, test) -> adds 'source' col -> samples 50k rows
#           source distribution: ~25k train / ~25k test
# Option B: load_train_only(train)         -> samples 50k rows
# Output  : ctx.df (50k x 47), sample.parquet saved to phase2_data_understanding/
ctx = run_api_phase2_1(ctx)

[20:22:30] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.api.run_facade_api - [run_api_phase2_1] start task=clustering run_id=20260502_202123
[20:22:30] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.pipeline.clustering_runner_pipeline - [run_clustering_pipeline_phase2_1] start run_id=20260502_202123 dataset_key=microsoft_security_incident option=A
[20:22:30] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.stage.stage2_understanding_runner_stage - [_resolve_load_options] multi source is: csv_combined
[20:22:30] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.stage.stage2_understanding_runner_stage - [_resolve_load_options] csv_params is:
[20:22:30] [DEBUG] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.stage.stage2_understanding_runner_stage - [_resolve_load_options] train=data/raw/train/GUIDE_Train.csv test=data/raw/test/GUIDE_Test.csv combine=True
[20:22:30] [DEBUG] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.stage.stage2_understanding_runner_stage - [run_initial_data_collection] _resolve_load_o

In [8]:
# Inspect loaded DataFrame — schema and source column
print(f"shape     : {ctx.df.shape}")
print(f"source_col: {ctx.source_col}")
print(f"is_option_a: {ctx.is_option_a}")
print(f"columns   : {list(ctx.df.columns)}")

shape     : (50000, 47)
source_col: csv_combined
is_option_a: True
columns   : ['Id', 'OrgId', 'IncidentId', 'AlertId', 'Timestamp', 'DetectorId', 'AlertTitle', 'Category', 'MitreTechniques', 'IncidentGrade', 'ActionGrouped', 'ActionGranular', 'EntityType', 'EvidenceRole', 'DeviceId', 'Sha256', 'IpAddress', 'Url', 'AccountSid', 'AccountUpn', 'AccountObjectId', 'AccountName', 'DeviceName', 'NetworkMessageId', 'EmailClusterId', 'RegistryKey', 'RegistryValueName', 'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId', 'ThreatFamily', 'FileName', 'FolderPath', 'ResourceIdName', 'ResourceType', 'Roles', 'OSFamily', 'OSVersion', 'AntispamDirection', 'SuspicionLevel', 'LastVerdict', 'CountryCode', 'State', 'City', 'source', 'Usage']


In [9]:
# First 5 rows with source column first — shows train/test origin per row
cols_preview = ["source"] + [c for c in ctx.df.columns if c != "source"]
ctx.df[cols_preview].head()

,source,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,...,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City,Usage
0,train,987842480303,1220,15534,664130,2024-06-03T12:46:12.000Z,33,22,InitialAccess,NaN,...,NaN,5,66,NaN,NaN,NaN,242,1445,10630,NaN
1,test,154618823937,1048,25063,564854,2024-06-06T22:07:37.000Z,5,80,SuspiciousActivity,NaN,...,NaN,5,66,NaN,NaN,NaN,242,1445,10630,Public
2,test,901943136916,103,328686,256721,2024-06-06T15:59:41.000Z,2,2,CommandAndControl,NaN,...,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630,Public
3,train,1365799603916,174,10388,98201,2024-06-11T12:29:02.000Z,201,65,Exfiltration,T1041,...,NaN,5,66,NaN,NaN,NaN,242,1445,10630,NaN
4,train,979252548312,722,53571,1357299,2024-06-03T09:18:48.000Z,5,5799,SuspiciousActivity,NaN,...,NaN,5,66,NaN,NaN,NaN,242,1445,10630,NaN


In [10]:
# 2.2. Sub-phase: Data Description
# Profiles ctx.df: schema, describe(), min/max/mean/std, cardinality, null counts
# Output : phase2_data_understanding/tables_png/*.png
#   describe.png              - full descriptive statistics
#   min_max_mean_std.png      - numeric columns range overview
#   schema_dtype_null_unique  - dtypes, null %, unique count per column
#   cardinality_top_50.png    - top 50 columns by unique value count
#   null_count.png            - null count sorted descending
ctx = run_api_phase2_2(ctx)

[20:22:38] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.api.run_facade_api - [run_api_phase2_2] start task=clustering run_id=20260502_202123
[20:22:38] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.pipeline.clustering_runner_pipeline - [run_clustering_pipeline_phase2_2] start run_id=20260502_202123 df.shape=(50000, 47)
[20:22:38] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.stage.stage2_understanding_runner_stage - [run_data_description] start shape=(50000, 47)
[20:22:40] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.reporting.artifacts_service_reporting - [artifacts] table png saved: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\out\runs\clustering\microsoft_security_incident\20260502_202123\stage2_understanding\step_2_2_describe_data\descriptive_statistics\describe.png
[20:22:42] [INFO] LOGGER_NAMESPACE_DATASCIENCE_01.crispdm.reporting.artifacts_service_reporting - [artifacts] table png saved: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incid

In [52]:
# 2.3. Sub-phase: Data Quality Verification
# Runs: missing value analysis, duplicate detection, business quality rules
# Option A only: drift detection per numeric column using PSI + KS test
#   PSI >= 0.20 or KS p-value < 0.05 -> column flagged as drifted
# Output : missingness_top.png, duplicates.png, drift_psi_summary.png
# Result : ctx.drift_detected = True/False  <- consumed by Stage 3
ctx = run_api_phase2_3(ctx)

[14:00:16] [INFO] MS_Security_Incident_Prediction.crispdm.api.run_facade_api - [run_phase2_3] start task=clustering run_id=20260501_135136 is_option_a=True
[14:00:16] [INFO] MS_Security_Incident_Prediction.crispdm.pipeline.clustering_runner_pipeline - [run_phase2_3] start run_id=20260501_135136 is_option_a=True
[14:00:16] [INFO] MS_Security_Incident_Prediction.crispdm.stage.stage2_understanding_runner_stage - [run_data_quality_verification] start is_option_a=True
[14:00:17] [INFO] MS_Security_Incident_Prediction.crispdm.reporting.artifacts_service_reporting - [artifacts] figure saved: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\out\runs\clustering\microsoft_security_incident\20260501_135136\stage2_understanding\figures\missingness_top.png
[14:00:18] [INFO] MS_Security_Incident_Prediction.crispdm.reporting.artifacts_service_reporting - [artifacts] table png saved: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\out\runs\clustering\mi

In [53]:
# Show drift report — which columns have significant distribution shift

parquet_dir = ctx.phase2_dir / "tables_png"
# The drift summary is in the PNG — read from stage_results instead
drift_info = ctx.stage_results.get("phase2_data_understanding", {})
print(f"drift_detected    : {drift_info.get('drift_detected')}")
print(f"n_drifted_cols    : {drift_info.get('n_drifted_cols')}")
print(f"n_analysed_cols   : {drift_info.get('n_analysed_cols')}")

drift_detected    : True
n_drifted_cols    : 12
n_analysed_cols   : 31


In [54]:
# 2.4. Sub-phase: Exploratory Data Analysis (EDA)
# Option A: generates train vs test histogram overlays per numeric column
#           visualises the drift detected in 2.3
# Option B: single histogram per numeric column
# Output : phase2_data_understanding/figures/hist_{col}_train.png
#          phase2_data_understanding/figures/hist_{col}_test.png
#          phase2_data_understanding/stage_report.json
ctx = run_api_phase2_4(ctx)

[14:02:05] [INFO] MS_Security_Incident_Prediction.crispdm.api.run_facade_api - [run_phase2_4] start task=clustering run_id=20260501_135136 drift_detected=True
[14:02:05] [INFO] MS_Security_Incident_Prediction.crispdm.pipeline.clustering_runner_pipeline - [run_phase2_4] start run_id=20260501_135136 drift_detected=True
[14:02:05] [INFO] MS_Security_Incident_Prediction.crispdm.stage.stage2_understanding_runner_stage - [run_exploratory_analysis] start df.shape=(200000, 47)
[14:02:06] [INFO] MS_Security_Incident_Prediction.crispdm.reporting.artifacts_service_reporting - [artifacts] figure saved: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\out\runs\clustering\microsoft_security_incident\20260501_135136\stage2_understanding\figures\hist_Id_train.png
[14:02:06] [INFO] MS_Security_Incident_Prediction.crispdm.reporting.artifacts_service_reporting - [artifacts] figure saved: K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\out\runs\clustering\m

In [58]:
#1. Ejecutar ahora (5 min):
import pandas as pd

ctx.df["IncidentGrade"].value_counts(normalize=True)
pd.to_datetime(ctx.df["Timestamp"]).dt.hour.value_counts().sort_index()



Timestamp
0      7606
1      7595
2      7298
3      6805
4      6832
5      6822
6      6833
7      6900
8      7200
9      7685
10     7975
11     8433
12     8930
13     9370
14     9612
15    10104
16    10158
17     9906
18     9950
19     9443
20     9253
21     8809
22     8647
23     7834
Name: count, dtype: int64

In [59]:

ctx.df["IncidentGrade"].value_counts(normalize=True)
ctx.df["IncidentGrade"].value_counts()

IncidentGrade
BenignPositive    84992
TruePositive      71263
FalsePositive     43217
Name: count, dtype: int64

In [10]:
############ 3. PHASE 3: Data Preparation ############

In [ ]:
# 3.1. Sub-phase: Data Selection

In [ ]:
# 3.2. Sub-phase: Data Cleaning

In [ ]:
# 3.3. Sub-phase: Data Transformation

In [ ]:
# 3.4. Sub-phase: Data Integration

In [ ]:
# 3.5. Sub-phase: Data Formatting

In [ ]:
############ 4. PHASE 4: Data Modeling ############

In [ ]:
# 4.1. Sub-phase: Model Technique Selection

In [ ]:
# 4.2. Sub-phase: Model Building

In [ ]:
# 4.3. Sub-phase: Test Design Generation

In [ ]:
# 4.4. Sub-phase: Model Evaluation (Technical)

In [ ]:
############ 5. PHASE 5: Evaluation & Interpretation ############

In [ ]:
# 5.1. Sub-phase: Knowledge Extraction & Interpretation

In [ ]:
# 5.2. Sub-phase: Business Results Evaluation

In [ ]:
# 5.3. Sub-phase: Process Audit / Review

In [ ]:
# 5.4. Sub-phase: Decision Making / Next Steps